# NIOS PDF Extraction — marker-pdf on Kaggle

**What this notebook does:**
1. Auto-discovers **all subjects** from the `nios-chapter-pdfs` dataset
2. Checks which chapters are already extracted (resumable — skips existing JSON files)
3. Tracks overall progress in `_progress.json`
4. Saves `Chapter N.json` per chapter → `/kaggle/working/extracted/class<level>/<subject>/chapters/`

**Setup:**
1. Add **`dipankaj/nios-chapter-pdfs`** as a dataset input
2. *(Optional)* Edit `SUBJECTS_TO_PROCESS` to target a single subject or leave as `"all"`
3. Enable GPU accelerator **(T4)** → **Run All** — it will only extract what isn't done yet

**Dataset layout:**
```
/kaggle/input/nios-chapter-pdfs/
  class10/<subject-id>/Chapter N.pdf
  class12/<subject-id>/Chapter N.pdf
```

**Output layout:**
```
/kaggle/working/extracted/
  _progress.json                       ← global progress tracker (all subjects)
  class10/<subject-id>/chapters/
    Chapter N.json
    _manifest.json
  class12/<subject-id>/chapters/
    Chapter N.json
    _manifest.json
```


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path
import json

# Which subjects to process:
#   "all"                        → every subject found in the dataset  (default)
#   "maths-12"                   → single subject slug
#   ["maths-12", "physics-12"]   → specific list
SUBJECTS_TO_PROCESS = "all"

_DATASET_ROOT = Path("/kaggle/input/nios-chapter-pdfs")
WORKING_DIR   = Path("/kaggle/working/extracted")
PROGRESS_FILE = WORKING_DIR / "_progress.json"

MARKER_FORMAT = "json"   # options: json, markdown, html
RESUME        = True     # skip chapters that are already extracted


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q --upgrade marker-pdf
# marker-pdf internally uses pypdfium2's v4 render API — v5 changed the signature
# pin to the last v4 release until marker-pdf ships a proper v5-compatible build
!pip install -q "pypdfium2==4.30.0"


In [ ]:
# ── Discover subjects & build work plan ───────────────────────────────────────
import json, re as _re
from pathlib import Path

WORKndbhdfghfgkdfR.mkd
# ── Load or init progress tracker ────────────────────────────────────────────
if PROGRESS_FILE.exists():
    progress = json.loads(PROGRESS_FILE.read_text())
    print(f"📂 Progress loaded — {len(progress['subjects'])} subject(s) tracked so far")
else:
    progress = {"last_updated": None, "subjects": {}}
    print("📂 No existing progress — starting fresh")

# ── Discover all subjects in the dataset ─────────────────────────────────────
def _discover_subjects():
    dirs = []
    for cls_dir in sorted(_DATASET_ROOT.glob("class*")):
        if cls_dir.is_dir():
            for subj_dir in sorted(cls_dir.iterdir()):
                if subj_dir.is_dir() and list(subj_dir.glob("*.pdf")):
                    dirs.append(subj_dir)
    return dirs

all_subject_dirs = _discover_subjects()

if SUBJECTS_TO_PROCESS == "all":
    subject_dirs = all_subject_dirs
elif isinstance(SUBJECTS_TO_PROCESS, str):
    subject_dirs = [d for d in all_subject_dirs if d.name == SUBJECTS_TO_PROCESS]
else:
    subject_dirs = [d for d in all_subject_dirs if d.name in SUBJECTS_TO_PROCESS]

def _sort_key(p):
    m = _re.search(r"\d+", p.stem)
    return (0 if p.stem.startswith("Chapter") else 1, int(m.group()) if m else 999)

# ── Build per-subject work plan ───────────────────────────────────────────────
print(f"\nDataset : {_DATASET_ROOT}  ({len(all_subject_dirs)} subjects found)\n")
print(f"{'Subject':<22}  {'PDFs':>4}  {'Done':>4}  {'Left':>4}  Status")
print("─" * 52)

subjects_info = []
for subj_dir in subject_dirs:
    subject_id  = subj_dir.name
    class_level = subject_id.rsplit("-", 1)[-1]
    out_dir     = WORKING_DIR / f"class{class_level}" / subject_id / "chapters"
    pdfs        = sorted(subj_dir.glob("*.pdf"), key=_sort_key)

    meta_path = subj_dir / "_manifest.json"
    subject_name = (
        json.loads(meta_path.read_text()).get("subject_name", subject_id)
        if meta_path.exists() else subject_id
    )

    done = set()
    if RESUME and out_dir.exists():
        done = {
            jf.stem for jf in out_dir.glob("*.json")
            if jf.name != "_manifest.json" and jf.stat().st_size > 100
        }

    pending = [p for p in pdfs if p.stem not in done]
    subjects_info.append({
        "id": subject_id, "name": subject_name, "class_level": class_level,
        "dir": subj_dir, "out_dir": out_dir,
        "pdfs": pdfs, "done": done, "pending": pending,
    })

    tag = "✅ complete" if not pending else ("⏳ partial" if done else "🆕 new")
    print(f"  {subject_id:<22}  {len(pdfs):>4}  {len(done):>4}  {len(pending):>4}  {tag}")

total_pending = sum(len(s["pending"]) for s in subjects_info)
print(f"\n{'Total chapters to extract:':<30} {total_pending}")


In [ ]:
# ── Extract with marker-pdf ───────────────────────────────────────────────────
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser
import datetime, json, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — extraction will be slow. Enable T4 in notebook settings.")

if total_pending == 0:
    print("\n✅ All chapters already extracted — nothing to do.")
else:
    print("🔧 Loading marker models...")
    config_parser = ConfigParser({"output_format": MARKER_FORMAT, "device": device})
    models = create_model_dict()
    print("✅ Models loaded\n")

    def extract_chapter(pdf_path: Path, out_dir: Path) -> bool:
        stem     = pdf_path.stem
        out_file = out_dir / f"{stem}.json"
        print(f"    ⚙️  {stem} ...")
        try:
            converter = PdfConverter(
                config=config_parser.generate_config_dict(),
                artifact_dict=models,
                processor_list=config_parser.get_processors(),
                renderer=config_parser.get_renderer(),
            )
            rendered = converter(str(pdf_path))
            text, _, _ = text_from_rendered(rendered)
            with open(out_file, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"       ✅ {out_file.stat().st_size / 1024:.0f} KB")
            return True
        except Exception as e:
            print(f"       ❌ {e}")
            return False

    def _checkpoint(subj, now):
        """Write per-subject manifest + flush progress tracker to disk."""
        if not subj["out_dir"].exists():
            return
        json_files = sorted(
            [f for f in subj["out_dir"].glob("*.json") if f.name != "_manifest.json"],
            key=_sort_key,
        )
        manifest = {
            "subject":       subj["id"],
            "subject_name":  subj["name"],
            "class_level":   subj["class_level"],
            "extracted_at":  now,
            "marker_format": MARKER_FORMAT,
            "chapters": [
                {"name": f.stem, "file": f.name, "size_bytes": f.stat().st_size}
                for f in json_files
            ],
        }
        with open(subj["out_dir"] / "_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)

        stats  = subj.get("stats", {})
        n_ok   = len(json_files)
        n_fail = stats.get("fail", 0)
        progress["subjects"][subj["id"]] = {
            "class_level":     subj["class_level"],
            "total_pdfs":      len(subj["pdfs"]),
            "extracted":       n_ok,
            "failed":          n_fail,
            "failed_chapters": stats.get("failed", []),
            "status":          "complete" if n_ok == len(subj["pdfs"]) and n_fail == 0 else "partial",
            "last_run":        now,
        }
        progress["last_updated"] = now
        PROGRESS_FILE.parent.mkdir(parents=True, exist_ok=True)
        with open(PROGRESS_FILE, "w") as f:
            json.dump(progress, f, indent=2)

    for si, subj in enumerate(subjects_info, 1):
        if not subj["pending"]:
            print(f"[{si}/{len(subjects_info)}] ⏭️  {subj['id']} — already complete")
            continue

        print(f"\n[{si}/{len(subjects_info)}] 📚 {subj['id']}"
              f"  ({len(subj['pending'])} to extract, {len(subj['done'])} already done)")
        subj["out_dir"].mkdir(parents=True, exist_ok=True)

        ok_count = fail_count = 0
        failed_chapters = []

        for pdf in subj["pdfs"]:
            if pdf.stem in subj["done"]:
                continue
            if extract_chapter(pdf, subj["out_dir"]):
                ok_count += 1
                subj["done"].add(pdf.stem)
            else:
                fail_count += 1
                failed_chapters.append(pdf.stem)

        subj["stats"] = {"ok": ok_count, "fail": fail_count, "failed": failed_chapters}
        print(f"  → ✅ {ok_count} extracted  ❌ {fail_count} failed")
        if failed_chapters:
            print(f"  → Failed: {failed_chapters}")

        # ── Checkpoint: save progress to disk after every subject ─────────────
        _checkpoint(subj, datetime.datetime.now(datetime.timezone.utc).isoformat())
        print(f"  💾 Progress saved")


In [ ]:
# ── Write manifests & update progress tracker ─────────────────────────────────
import datetime

now = datetime.datetime.now(datetime.timezone.utc).isoformat()

for subj in subjects_info:
    if not subj["out_dir"].exists():
        continue

    json_files = sorted(
        [f for f in subj["out_dir"].glob("*.json") if f.name != "_manifest.json"],
        key=_sort_key,
    )

    manifest = {
        "subject":       subj["id"],
        "subject_name":  subj["name"],
        "class_level":   subj["class_level"],
        "extracted_at":  now,
        "marker_format": MARKER_FORMAT,
        "chapters": [
            {"name": f.stem, "file": f.name, "size_bytes": f.stat().st_size}
            for f in json_files
        ],
    }
    with open(subj["out_dir"] / "_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    stats  = subj.get("stats", {})
    n_ok   = len(json_files)
    n_fail = stats.get("fail", 0)
    progress["subjects"][subj["id"]] = {
        "class_level":     subj["class_level"],
        "total_pdfs":      len(subj["pdfs"]),
        "extracted":       n_ok,
        "failed":          n_fail,
        "failed_chapters": stats.get("failed", []),
        "status":          "complete" if n_ok == len(subj["pdfs"]) and n_fail == 0 else "partial",
        "last_run":        now,
    }

progress["last_updated"] = now
with open(PROGRESS_FILE, "w") as f:
    json.dump(progress, f, indent=2)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Subject':<22}  {'Total':>5}  {'Done':>5}  {'Fail':>5}  Status")
print("─" * 56)
for sid, info in progress["subjects"].items():
    icon = "✅" if info["status"] == "complete" else "⚠️ "
    print(f"  {sid:<22}  {info['total_pdfs']:>5}  {info['extracted']:>5}  {info['failed']:>5}  {icon}")
print(f"\n📋 Progress saved → {PROGRESS_FILE}")


In [ ]:
# ── Preview one chapter from any extracted subject ────────────────────────────
_all_chapter_files = []
for _d in sorted(WORKING_DIR.glob("class*/*/chapters")):
    _all_chapter_files.extend(sorted(
        _d.glob("Chapter *.json"),
        key=lambda p: int(m.group()) if (m := _re.search(r"\d+", p.stem)) else 999,
    ))

if _all_chapter_files:
    sample = _all_chapter_files[0]
    with open(sample) as f:
        data = json.load(f)

    # show which subject/chapter this is
    rel = sample.relative_to(WORKING_DIR)
    print(f"Preview : {rel}")
    print(f"Keys    : {list(data.keys())}")
    print()
    blocks = data.get("blocks", [])
    print(f"Total blocks: {len(blocks)}\nFirst 5:")
    for b in blocks[:5]:
        btype = b.get("block_type", b.get("type", "unknown"))
        raw_lines = b.get("text_lines", []) or b.get("lines", [])
        text_preview = " ".join(
            line.get("text", "") if isinstance(line, dict) else str(line)
            for line in (raw_lines[:2] if raw_lines else [])
        )[:120]
        print(f"  [{btype}] {text_preview}")
else:
    print("No extracted chapters found yet — run cells 4 & 5 first.")
